In [1]:
# ================================
# BASELINE MODEL WITH CHRONOLOGICAL SPLITS
# ================================

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
import joblib
import os

print("🤖 BUILDING BASELINE MODEL WITH CHRONOLOGICAL SPLITS")
print("=" * 60)

# ================================
# LOAD CHRONOLOGICAL SPLITS
# ================================

splits_dir = '../DATA/splits'
print("📂 Loading chronological split files...")

# Load features
X_train = pd.read_csv(f'{splits_dir}/X_train.csv')
X_val = pd.read_csv(f'{splits_dir}/X_val.csv') 
X_test = pd.read_csv(f'{splits_dir}/X_test.csv')

# Load targets  
y_train = pd.read_csv(f'{splits_dir}/y_train.csv').iloc[:, 0]  # Convert to Series
y_val = pd.read_csv(f'{splits_dir}/y_val.csv').iloc[:, 0]
y_test = pd.read_csv(f'{splits_dir}/y_test.csv').iloc[:, 0]

print(f"✅ Train: X{X_train.shape}, y{y_train.shape}")
print(f"✅ Val:   X{X_val.shape}, y{y_val.shape}")  
print(f"✅ Test:  X{X_test.shape}, y{y_test.shape}")

# ================================
# FEATURE SCALING (OPTIONAL BUT RECOMMENDED)
# ================================

print("\n🔧 Scaling features...")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("✅ Features scaled using StandardScaler")

# ================================
# TRAIN BASELINE MODEL
# ================================

print("\n🌳 Training Random Forest baseline...")

# Random Forest - good baseline for tabular data
rf_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

# Train on scaled features
rf_model.fit(X_train_scaled, y_train)
print("✅ Model trained successfully!")

# ================================
# MAKE PREDICTIONS
# ================================

print("\n🔮 Making predictions...")

# Predictions on all sets
y_train_pred = rf_model.predict(X_train_scaled)
y_val_pred = rf_model.predict(X_val_scaled)
y_test_pred = rf_model.predict(X_test_scaled)

print("✅ Predictions generated for train/val/test sets")

# ================================
# EVALUATE MODEL PERFORMANCE
# ================================

def evaluate_predictions(y_true, y_pred, set_name):
    """Calculate and print evaluation metrics"""
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    
    print(f"\n📊 {set_name} Set Performance:")
    print(f"   MAE:  {mae:.4f}")
    print(f"   MSE:  {mse:.4f}") 
    print(f"   RMSE: {rmse:.4f}")
    print(f"   R²:   {r2:.4f}")
    
    return {"MAE": mae, "MSE": mse, "RMSE": rmse, "R2": r2}

print("\n🎯 MODEL EVALUATION")
print("=" * 40)

# Evaluate on all sets
train_metrics = evaluate_predictions(y_train, y_train_pred, "TRAIN")
val_metrics = evaluate_predictions(y_val, y_val_pred, "VALIDATION")
test_metrics = evaluate_predictions(y_test, y_test_pred, "TEST")

# ================================
# CHECK FOR OVERFITTING
# ================================

print(f"\n🔍 OVERFITTING CHECK:")
print(f"   Train RMSE: {train_metrics['RMSE']:.4f}")
print(f"   Val RMSE:   {val_metrics['RMSE']:.4f}")
print(f"   Test RMSE:  {test_metrics['RMSE']:.4f}")

val_train_diff = val_metrics['RMSE'] - train_metrics['RMSE']
if val_train_diff > 0.05:  # 5% threshold
    print(f"⚠️  Possible overfitting detected (Val-Train RMSE diff: {val_train_diff:.4f})")
else:
    print(f"✅ No significant overfitting (Val-Train RMSE diff: {val_train_diff:.4f})")

# ================================
# SAVE MODEL AND SCALER
# ================================

print(f"\n💾 Saving model artifacts...")

models_dir = '../models'
os.makedirs(models_dir, exist_ok=True)

# Save model and scaler
joblib.dump(rf_model, f'{models_dir}/baseline_rf_model.pkl')
joblib.dump(scaler, f'{models_dir}/feature_scaler.pkl')

print(f"✅ Saved baseline_rf_model.pkl ({os.path.getsize(f'{models_dir}/baseline_rf_model.pkl')/1024/1024:.1f} MB)")
print(f"✅ Saved feature_scaler.pkl ({os.path.getsize(f'{models_dir}/feature_scaler.pkl')/1024:.1f} KB)")

# ================================
# FEATURE IMPORTANCE ANALYSIS
# ================================

print(f"\n🔍 TOP 10 MOST IMPORTANT FEATURES:")
print("-" * 40)

# Get feature importance
feature_importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

for i, (feature, importance) in enumerate(feature_importance.head(10).values):
    print(f"{i+1:2d}. {feature:25s} {importance:.4f}")

# ================================
# SUMMARY
# ================================

print(f"\n" + "="*60)
print(f"🎉 BASELINE MODEL TRAINING COMPLETE!")
print(f"="*60)
print(f"✅ Model Type: Random Forest Regressor")
print(f"✅ Split Type: Chronological (70/15/15)")
print(f"✅ Features: {X_train.shape[1]} engineered features")
print(f"✅ Test RMSE: {test_metrics['RMSE']:.4f}")
print(f"✅ Test R²: {test_metrics['R2']:.4f}")
print(f"✅ Ready for advanced modeling!")

# ================================
# PREDICTIONS PREVIEW
# ================================

print(f"\n📋 PREDICTIONS PREVIEW (First 10 test samples):")
preview_df = pd.DataFrame({
    'Actual_SOH': y_test.iloc[:10].values,
    'Predicted_SOH': y_test_pred[:10],
    'Error': y_test.iloc[:10].values - y_test_pred[:10]
})
preview_df['Error_Abs'] = np.abs(preview_df['Error'])
print(preview_df.round(4))

🤖 BUILDING BASELINE MODEL WITH CHRONOLOGICAL SPLITS
📂 Loading chronological split files...
✅ Train: X(10544, 11), y(10544,)
✅ Val:   X(2260, 11), y(2260,)
✅ Test:  X(2260, 11), y(2260,)

🔧 Scaling features...
✅ Features scaled using StandardScaler

🌳 Training Random Forest baseline...
✅ Model trained successfully!

🔮 Making predictions...
✅ Predictions generated for train/val/test sets

🎯 MODEL EVALUATION

📊 TRAIN Set Performance:
   MAE:  0.0001
   MSE:  0.0000
   RMSE: 0.0011
   R²:   1.0000

📊 VALIDATION Set Performance:
   MAE:  0.0779
   MSE:  0.0083
   RMSE: 0.0913
   R²:   -3.0882

📊 TEST Set Performance:
   MAE:  0.2339
   MSE:  0.0597
   RMSE: 0.2443
   R²:   -32.3411

🔍 OVERFITTING CHECK:
   Train RMSE: 0.0011
   Val RMSE:   0.0913
   Test RMSE:  0.2443
⚠️  Possible overfitting detected (Val-Train RMSE diff: 0.0902)

💾 Saving model artifacts...
✅ Saved baseline_rf_model.pkl (9.8 MB)
✅ Saved feature_scaler.pkl (1.4 KB)

🔍 TOP 10 MOST IMPORTANT FEATURES:
-----------------------